<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 05. A3C

## Asynchronous Methods for Deep Reinforcement Learning

- **원 논문:** [Asynchronous Methods for Deep Reinforcement Learning](https://proceedings.mlr.press/v48/mniha16.html)
- **저자 / 발표:** Volodymyr Mnih et al. · ICML (2016)
- **난이도 / 예상 시간:** 고급 · 약 65분
- **선수 지식:** actor-critic, n-step return, entropy regularization
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 공유 actor-critic에 여러 seed worker rollout의 n-step advantage를 순차 적용해 A3C update를 재현한다.

**축소하거나 생략한 부분:** 논문은 다중 CPU thread와 Atari/Labyrinth 환경을 사용한다. 결정적 검증을 위해 worker를 순차 실행하지만 shared parameters, n-step return, entropy objective는 동일하다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3, n-step Q-learning paragraph | bootstrapped n-step return | k rewards와 terminal이 아닐 때 V(s_{t+k})를 합친다. |
| §4, ‘Asynchronous advantage actor-critic’ | advantage actor and value losses | R_t-V(s_t)로 policy/value를 동시에 갱신한다. |
| §4 entropy equation; Supplementary Algorithm S2 | entropy-regularized shared-worker loop | 여러 actor-learners의 탐색과 shared parameter update를 축소 재현한다. |

## 핵심 재현

        §4의 advantage $R_t-V(s_t)$와 entropy bonus를 사용합니다. 실제 thread 경쟁은
        비결정적이므로 네 개 seed worker가 같은 모델에 차례로 gradient를 적용합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(451)
np.random.seed(451)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
\mathcal L=-\log\pi(a_t|s_t)(R_t-V(s_t))+c_v(R_t-V(s_t))^2-c_e\mathcal H(\pi(\cdot|s_t))
$$

- **기호와 역할:** $R_t$는 n-step bootstrap return, $V$는 critic, $\mathcal H$는 worker 탐색을 유지하는 entropy입니다.
- **shape/state 계약:** `state [T] → policy logits [T, A], value/return/advantage [T]`
- **논문 위치:** `§3, n-step Q-learning paragraph`의 **bootstrapped n-step return**
- **코드 Task:** n-step return, shared actor-critic loss, worker rollout update를 구현합니다.
- **학습·평가 흐름:** worker rollout → bootstrap return → advantage → shared update → greedy 평가
- **원문 대비 한계:** 논문은 다중 CPU thread와 Atari/Labyrinth 환경을 사용한다. 결정적 검증을 위해 worker를 순차 실행하지만 shared parameters, n-step return, entropy objective는 동일하다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class SharedActorCriticAgent(nn.Module):
    """A3C의 핵심 학습·평가 경로. 입출력 계약: state [T] → policy logits [T, A],
    value/return/advantage [T]."""

    def __init__(self, network, optimizer, value_weight=0.5, entropy_weight=0.01):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.network = network
        self.optimizer = optimizer
        self.value_weight = value_weight
        self.entropy_weight = entropy_weight

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {
            "value_weight": self.value_weight,
            "entropy_weight": self.entropy_weight,
            "worker_mode": "deterministic-sequential",
        }

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: state [T] → policy logits [T, A],
        value/return/advantage [T]."""
        policy_logits, value = self.network(state)
        probability = torch.softmax(policy_logits, dim=-1)
        return (probability, value)

    def loss(self, log_probabilities, values, returns, entropies):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        advantage = returns - values
        policy_loss = -(log_probabilities * advantage.detach()).mean()
        value_loss = advantage.square().mean()
        entropy_bonus = entropies.mean()
        objective = policy_loss + self.value_weight * value_loss
        objective = objective - self.entropy_weight * entropy_bonus
        return objective

    def update(self, log_probabilities, values, returns, entropies):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(log_probabilities, values, returns, entropies)
        self.optimizer.zero_grad()
        objective.backward()
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), 5.0)
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, state, reference_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            probability, value = self.forward(state)
            greedy_action = probability.argmax(dim=1)
            optimal_action = reference_q.to(DEVICE).argmax(dim=1)
            agreement = (greedy_action == optimal_action).float().mean()
        return {"greedy_agreement": float(agreement), "mean_value": float(value.mean())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, n-step Q-learning paragraph` — **bootstrapped n-step return**
- **구현 이유:** k rewards와 terminal이 아닐 때 V(s_{t+k})를 합친다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def nstep_returns(reward_list, bootstrap, discount):
    values = []
    running = bootstrap
    for reward in reversed(reward_list):
        running = torch.as_tensor(reward, device=bootstrap.device) + discount * running
        values.append(running)
    return torch.stack(list(reversed(values)))


out = nstep_returns([1.0, 2.0], torch.tensor(3.0), 0.5)
assert torch.allclose(out, torch.tensor([2.75, 3.5]))

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§4, ‘Asynchronous advantage actor-critic’` — **advantage actor and value losses**
- **구현 이유:** R_t-V(s_t)로 policy/value를 동시에 갱신한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_states, 32), nn.ReLU())
        self.actor = nn.Linear(32, n_actions)
        self.critic = nn.Linear(32, 1)

    def forward(self, s):
        h = self.body(encode_device(s))
        return self.actor(h), self.critic(h).squeeze(-1)


model = ActorCritic().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=0.008)
history = []

portfolio_agent = SharedActorCriticAgent(model, opt)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
for round_id in range(100):
    for worker in range(4):
        generator = torch.Generator().manual_seed(452 + 1000 * round_id + worker)
        state = start_state
        log_probabilities = []
        rollout_values = []
        entropies = []
        rollout_rewards = []
        done = False
        for _ in range(2 * n_states + 2):
            probabilities, value = portfolio_agent.forward(torch.tensor([state]))
            probability = probabilities[0]
            action = int(
                torch.multinomial(probability.detach().cpu(), 1, generator=generator)
            )
            next_state, reward, done = env_step(state, action)
            log_probabilities.append(torch.log(probability[action].clamp_min(1e-8)))
            entropies.append(
                -(probability * torch.log(probability.clamp_min(1e-8))).sum()
            )
            rollout_values.append(value[0])
            rollout_rewards.append(reward)
            state = next_state
            if done:
                break
        bootstrap = torch.tensor(0.0, device=DEVICE)
        if not done:
            bootstrap = portfolio_agent.forward(torch.tensor([state]))[1][0].detach()
        returns = nstep_returns(rollout_rewards, bootstrap, gamma)
        history.append(
            portfolio_agent.update(
                torch.stack(log_probabilities),
                torch.stack(rollout_values),
                returns,
                torch.stack(entropies),
            )
        )
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§4 entropy equation; Supplementary Algorithm S2` — **entropy-regularized shared-worker loop**
- **구현 이유:** 여러 actor-learners의 탐색과 shared parameter update를 축소 재현한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(torch.arange(n_states), q_star)
assert portfolio_metrics

with torch.no_grad():
    logits, values = model(torch.arange(n_states))
    policy = logits.softmax(1)
    nonterminal = torch.arange(n_states, device=DEVICE) != terminal_state
    agreement = float(
        (policy.argmax(1)[nonterminal] == q_star.to(DEVICE).argmax(1)[nonterminal])
        .float()
        .mean()
    )
state = start_state
total = 0.0
seen = set()
for _ in range(2 * n_states + 2):
    action = int(policy[state].argmax())
    ns, r, done = env_step(state, action)
    total += r
    state = ns
    if done or (state, action) in seen:
        break
    seen.add((state, action))
print(f"greedy return={total:.3f}, optimal-action agreement={agreement:.3f}")
assert math.isfinite(total) and agreement >= 0.5
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(history)
axes[0].set_title("shared-worker loss")
axes[1].imshow(policy.detach().cpu().T, aspect="auto", vmin=0, vmax=1)
axes[1].set_title("actor policy")
fig.tight_layout()
plt.show()

실제 A3C는 서로 다른 환경 상태와 stale parameter에서 계산한 gradient가 비동기적으로 공유 파라미터에 도착합니다. 여기서는 그 decorrelation 효과 대신 서로 다른 seed rollout과 shared update 식만 보존합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3, n-step Q-learning paragraph`의 **bootstrapped n-step return**
- **핵심 식:**

  $$
  \mathcal L=-\log\pi(a_t|s_t)(R_t-V(s_t))+c_v(R_t-V(s_t))^2-c_e\mathcal H(\pi(\cdot|s_t))
  $$

- **입출력 shape:** `state [T] → policy logits [T, A], value/return/advantage [T]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `SharedActorCriticAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. k rewards와 terminal이 아닐 때 V(s_{t+k})를 합친다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 다중 CPU thread와 Atari/Labyrinth 환경을 사용한다. 결정적 검증을 위해 worker를 순차 실행하지만 shared parameters, n-step return, entropy objective는 동일하다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.